In [1]:
import feedparser

In [2]:
feed = feedparser.parse("https://feeds.simplecast.com/dxZsm5kX")

In [3]:
EXPLICIT_CHOICES = (
    (1, "yes"), 
    (2, "no"), 
    (3, "clean")
)

CHANNEL_TYPE_CHOICES = (
    (1, "episodic"), 
    (2, "serial")
)

ITEM_TYPE_CHOICES = (
    (1, "full"), 
    (2, "trailer"),
    (3, "bonus"),
)

explicit_dict = dict((choice[1], choice[0])for choice in EXPLICIT_CHOICES)
channel_type_dict = dict((choice[1], choice[0])for choice in CHANNEL_TYPE_CHOICES)
item_type_dict = dict((choice[1], choice[0])for choice in ITEM_TYPE_CHOICES)

In [11]:
feed.feed["itunes_explicit"]

True

In [12]:
new_feed = {
    "rss": next(item for item in feed.feed.links if item["rel"] == "self")["href"],
    "title": feed.feed.title,
    "link": feed.feed.link,
    "image": feed.feed.image.href,
    "language": feed.feed.language,
    "copyright": feed.feed.copyright,
    "subtitle": feed.feed.subtitle,
    "author": feed.feed.author,
    "summary": feed.feed.summary,
    "description": feed.feed.description,
    "owner": feed.feed.publisher_detail.email,
    "categories": ", ".join([cat.term for cat in feed.feed.tags]),
    "description": feed.feed.description,
}

if feed.feed.itunes_type is not None:
    new_feed["type"] = channel_type_dict.get(feed.feed.itunes_type.lower())

if feed.feed.itunes_explicit is not None:
    if feed.feed.itunes_explicit:
        new_feed["explicit"] = explicit_dict.get("yes")
    else:
        new_feed["explicit"] = explicit_dict.get("no")


new_feed

{'rss': 'https://feeds.simplecast.com/dxZsm5kX',
 'title': 'Pod Save America',
 'link': 'https://crooked.com/',
 'image': 'https://image.simplecastcdn.com/images/9aa1e238-cbed-4305-9808-c9228fc6dd4f/eb7dddd4-ecb0-444c-b379-f75d7dc6c22b/3000x3000/uploads-2f1595947484360-nc4atf9w7ur-dbbaa7ee07a1ee325ec48d2e666ac261-2fpodsave100daysfinal1800.jpg?aid=rss_feed',
 'language': 'en',
 'copyright': '© Crooked Media. All Rights Reserved.',
 'subtitle': 'Four former aides to President Obama—Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor—are joined by journalists, politicians, activists, and more for a no-b******t conversation about politics. They cut through the noise to break down the week’s news, and help people figure out what matters and how they can help.\xa0You can listen to new episodes twice a week on Tuesdays and Thursdays.',
 'author': 'Crooked Media',
 'summary': 'Four former aides to President Obama—Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor—are joined by journalis

In [19]:
feed.entries[0]

{'id': 'fc3a9b49-41b4-4eb2-b1eb-567da687da90',
 'guidislink': False,
 'title': '“Tuck Around and Find Out.”',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://feeds.simplecast.com/dxZsm5kX',
  'value': '“Tuck Around and Find Out.”'},
 'summary': 'Tucker Carlson hates Donald Trump but loves his insurrectionists, Democratic pollster Celinda Lake stops by to talk about Joe Biden’s new economic plan, and Chief Take Officer Elijah Cone joins for a game of Take Take Don’t Tell Me.',
 'summary_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://feeds.simplecast.com/dxZsm5kX',
  'value': 'Tucker Carlson hates Donald Trump but loves his insurrectionists, Democratic pollster Celinda Lake stops by to talk about Joe Biden’s new economic plan, and Chief Take Officer Elijah Cone joins for a game of Take Take Don’t Tell Me.'},
 'published': 'Thu, 9 Mar 2023 22:51:20 +0000',
 'published_parsed': time.struct_time(tm_year=2023, tm_mon=3, tm_mday=9, tm_hou

In [21]:
episodes = []
for entry in feed.entries:
    episode = {
        "title": entry.title,
        "author": entry.author,
        "guid": entry.guid,
        "pub_date": entry.published,
        "episode_type": item_type_dict.get(entry.itunes_episodetype.lower()),
        "duration": entry.itunes_duration,
        "audio_link": next(item for item in entry.links if item["rel"] == "enclosure")["href"],
    }

    if "summary" in entry.keys() and len(entry.summary) > 0:
        episode["summary"] = entry.summary

    if "description" in entry.keys() and len(entry.description) > 0:
        episode["description"] = entry.description

    if "subtitle" in entry.keys() and len(entry.subtitle) > 0:
        episode["subtitle"] = entry.subtitle

    if "link" in entry.keys():
        episode["link"] = entry.link

    if "itunes_episode" in entry.keys():
        episode["episode_num"] = entry.itunes_episode

    if "itunes_explicit" in entry.keys():
        if entry.itunes_explicit:
            episode["explicit"] = explicit_dict.get("yes")
        else:
            episode["explicit"] = explicit_dict.get("no")

    if "image" in entry.keys():
        episode["image"] = entry.image.href

    if "itunes_season" in entry.keys():
        episode["season"] = entry.itunes_season

    
    episodes.append(episode)


In [22]:
new_feed["audioitem_set"] = episodes
new_feed

{'rss': 'https://feeds.simplecast.com/dxZsm5kX',
 'title': 'Pod Save America',
 'link': 'https://crooked.com/',
 'image': 'https://image.simplecastcdn.com/images/9aa1e238-cbed-4305-9808-c9228fc6dd4f/eb7dddd4-ecb0-444c-b379-f75d7dc6c22b/3000x3000/uploads-2f1595947484360-nc4atf9w7ur-dbbaa7ee07a1ee325ec48d2e666ac261-2fpodsave100daysfinal1800.jpg?aid=rss_feed',
 'language': 'en',
 'copyright': '© Crooked Media. All Rights Reserved.',
 'subtitle': 'Four former aides to President Obama—Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor—are joined by journalists, politicians, activists, and more for a no-b******t conversation about politics. They cut through the noise to break down the week’s news, and help people figure out what matters and how they can help.\xa0You can listen to new episodes twice a week on Tuesdays and Thursdays.',
 'author': 'Crooked Media',
 'summary': 'Four former aides to President Obama—Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor—are joined by journalis

In [23]:
# convert new_feed dictionary to jscon string
import json
s = json.dumps(new_feed)

In [24]:
print(s)

{"rss": "https://feeds.simplecast.com/dxZsm5kX", "title": "Pod Save America", "link": "https://crooked.com/", "image": "https://image.simplecastcdn.com/images/9aa1e238-cbed-4305-9808-c9228fc6dd4f/eb7dddd4-ecb0-444c-b379-f75d7dc6c22b/3000x3000/uploads-2f1595947484360-nc4atf9w7ur-dbbaa7ee07a1ee325ec48d2e666ac261-2fpodsave100daysfinal1800.jpg?aid=rss_feed", "language": "en", "copyright": "\u00a9 Crooked Media. All Rights Reserved.", "subtitle": "Four former aides to President Obama\u2014Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor\u2014are joined by journalists, politicians, activists, and more for a no-b******t conversation about politics. They cut through the noise to break down the week\u2019s news, and help people figure out what matters and how they can help.\u00a0You can listen to new episodes twice a week on Tuesdays and Thursdays.", "author": "Crooked Media", "summary": "Four former aides to President Obama\u2014Jon Favreau, Jon Lovett, Dan Pfeiffer and Tommy Vietor\u201

In [ ]:
%cd ..